# Lesson 7 : Workflows

Up until now, we have built agents with language models, tools, and messages.<br>
This type of simple agent (i.e., a single agent) essentially performs to call LLMs or tools until it reaches to the goal (such as, [ReAct](https://tsmatz.wordpress.com/2023/03/07/react-with-openai-gpt-and-langchain/)-style pattern), but various processing patterns (such as, workflow pattern, etc) will be required in production. (See below picture.)

![Flow patterns](./assets/flow_patterns.png)

Using built-in process builders in Agent Framework, you can build various processing patterns to orchestrate your existing agents.  
```WorkflowBuilder``` is a generic builder for a flexible workflow graph, and other builders (```SequentialBuilder```, ```ConcurrentBuilder```, ```GroupChatBuilder```, ```HandoffBuilder```, and ```MagenticBuilder```) handles a variety of processing patterns depending on your design patterns, which are all built with ```WorkflowBuilder```.

> Note : By dividing the roles among multiple agents in complex tasks, the context becomes clear, and accuracy is expected to improve. (Also, manageability will increase.)  
> Learning a variety of multi-agent design patterns (orchestration patterns) is out of scope in this repository. (See [official document](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns) for this topics.)

## 1. Build and run a simple workflow

In this example, we build the following brief sequential flow with a generic ```WorkflowBuilder```, in order for learning the backend in workflow processing.

1. Create a plan for the trip.
2. Revise the generated plan.

First we initialize the client as follows.

In [1]:
from agent_framework.azure import AzureAIClient
from azure.identity.aio import AzureCliCredential

credential = AzureCliCredential()
client = AzureAIClient(credential=credential)

Now let's build a workflow and run as follows.  
As I have mentioned above, you can use ```SequentialBuilder``` (instead of generic ```WorkflowBuilder```) for this type of simple sequential flows, but I'll implement it using generic ```WorkflowBuilder``` class in this example.

In workflow on Agent Framework, first we register ```Executor``` as building blocks.  
In this example, we use the following 3 ```Executor``` - two is ```AgentExecutor``` (which is generated and registered by the following ```register_agent()```) and one is generic ```Executor``` (which is registered by the following ```register_executor()```).

- PlanningAgent : This is an executor by LLM agent to generate a plan.
- ReviseAgent : This is an executor by LLM agent to revise a plan.
- FinalResponseGenerator : This executor generates the final response and finalize the workflow. (The workflow's instance becomes idle after this executor is run.) In this example, we build a list of all messages for the final response.

As you can see in below code, a generic ```Executor``` is processed in a class method (function) decorated by ```@handler```. (You can also define multiple handlers depending on arguments, in a single ```Executor```.)  
All executors are connected by workflow context, and you can pass result (output) to another executor by using this context. (See the following ```generate_final_respose()``` handler.)

> Note : It is not recommended to split a simple task into multiple agents (such like, this example), because it might degrade performance.  
> This example is for demo purpose.

In [2]:
from agent_framework import ChatAgent, ChatMessage
from agent_framework import (
    WorkflowBuilder,
    Executor,
    AgentExecutorResponse,
    WorkflowContext,
    handler
)

class ResponseGenerator(Executor):
    @handler
    async def generate_final_respose(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext[list[ChatMessage]]
    ) -> None:
        # in this example, we send full conversation as a final result.
        await ctx.yield_output(list(response.full_conversation))

workflow = (
    WorkflowBuilder()
    .register_agent(
        lambda: ChatAgent(
            name="PlanningAgent",
            chat_client=client,
            instructions="ユーザーの計画作成を支援し、箇条書きの 5 項目に簡潔にまとめます。",  # "your task is to help users plan and concisely summarize it in five bullet points."
        ),
        name="PlanningAgent",
        output_response=True,
    )
    .register_agent(
        lambda: ChatAgent(
            name="ReviseAgent",
            chat_client=client,
            instructions="与えられた計画を確認して、より洗練させます。",  # "your task is to review the plan you receive and to refine it further."
        ),
        name="ReviseAgent",
        output_response=False,
    )
    .register_executor(
        lambda: ResponseGenerator(id="final_response_generator"),
        name="FinalResponseGenerator"
    )
    .add_edge("PlanningAgent", "ReviseAgent")
    .add_edge("ReviseAgent", "FinalResponseGenerator")
    .set_start_executor("PlanningAgent")
    .build()
)

events = await workflow.run("大阪の旅行計画を作成してください。")  # "create a plan for my travel in Osaka"

As you saw above, all messages are returned in this workflow.  
Now we output the returned messages as follows.

In [3]:
from agent_framework import Role

outputs = events.get_outputs()
messages = outputs[1]
for i, (msg) in enumerate(messages, start=1):
    author_name = msg.author_name
    if author_name is None:
        author_name = "assistant" if msg.role == Role.ASSISTANT else "user"
    print("----------------------------------")
    print(f"[{i:02d}: {author_name}]")
    print(msg.text)

----------------------------------
[01: user]
大阪の旅行計画を作成してください。
----------------------------------
[02: PlanningAgent]
- **日程・目的の設定**：1泊2日/2泊3日など日数、目的（食い倒れ・観光・USJ・買い物・温泉）を決める  
- **エリア別の回り方**：①難波〜道頓堀（グルメ・夜景）②梅田（買い物・展望）③大阪城（歴史）④新世界（通天閣）を軸に動線を組む  
- **モデル行程案**：1日目：道頓堀→心斎橋→新世界（串カツ）／2日目：大阪城→梅田（空中庭園）→帰路（時間があれば黒門市場）  
- **食事候補の押さえ**：たこ焼き・お好み焼き・串カツ・うどん・寿司（人気店は事前予約/ピーク時間回避）  
- **移動・宿・チケット**：宿は「難波/梅田」中心が便利、移動は地下鉄・ICカード、USJや展望台は事前チケット購入を検討（混雑対策）
----------------------------------
[03: ReviseAgent]
- **日程・目的を決める**：何泊か（例：1泊2日/2泊3日）＋目的（食い倒れ、USJ、買い物、歴史、家族向け等）を確定  
- **拠点エリア選び**：移動重視なら「難波（ミナミ）」or「梅田（キタ）」に宿を取る（初大阪は難波が観光・グルメに強い）  
- **モデル行程（1泊2日）**：1日目：道頓堀〜心斎橋→黒門市場→新世界（通天閣・串カツ）／2日目：大阪城→梅田（グランフロント・空中庭園）→帰路  
- **食の候補を事前に整理**：たこ焼き・お好み焼き・串カツ・うどん・寿司（人気店は予約 or 時間ずらしで混雑回避）  
- **移動・チケットの準備**：地下鉄＋ICカード中心、USJや展望台などは事前チケット購入、土日や繁忙期は時間指定を優先


## 2. Run a workflow with streaming

You can also run workflow with streaming.  
This scenario is useful when you handle checkpointing (pause / resume), human-in-the-loop (HITL), etc. (These topics will be discussed later.)

In [4]:
from agent_framework import WorkflowOutputEvent

async for event in workflow.run_stream("大阪の旅行計画を作成してください。"):  # "create a plan for my travel in Osaka"
    print(f"[{event}]")
    # if isinstance(event, WorkflowOutputEvent):
    #     print("----------------------------------")
    #     print(event.data)

[WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)]
[WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)]
[ExecutorInvokedEvent(executor_id=PlanningAgent, data=大阪の旅行計画を作成してください。)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=-)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages= **)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=前)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=提)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=（)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=まず)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=決)]
[AgentRunUpdateEvent(executor_id=PlanningAgent, messages=め)]
[AgentRunUpdateE

## 3. Checkpoint

When you run a long-running workflow, it might need to suspend the running flow and restart it later.  
With checkpoint in workflow on Agent Framework, you can save (serialize) the state in the storage, and restore (deserialize) its state to resume the suspended instance.

In this example, we explore checkpoint using ```InMemoryCheckpointStorage``` (non-persistent storage) for demo purpose.

Checkpoint is created after each super step iteration.  
In this example, therefore, we suspend workflow when the first ```SuperStepCompletedEvent``` is captured. And we then restore the checkpoint and resume workflow.

First we generate a workflow again, with checkpoint enabled.

> Note : When you need to restore some data in your custom ```Executor```, please implement ```on_checkpoint_save()``` and ```on_checkpoint_restore()``` methods (override methods) in your executor class.  
> (This example doesn't have such data.)

In [5]:
from agent_framework import InMemoryCheckpointStorage

checkpoint_storage = InMemoryCheckpointStorage()

workflow_builder = (
    WorkflowBuilder()
    .register_agent(
        lambda: ChatAgent(
            name="PlanningAgent",
            chat_client=client,
            instructions="ユーザーの計画作成を支援し、箇条書きの 5 項目に簡潔にまとめます。",  # "your task is to help users plan and concisely summarize it in five bullet points."
        ),
        name="PlanningAgent",
        output_response=True,
    )
    .register_agent(
        lambda: ChatAgent(
            name="ReviseAgent",
            chat_client=client,
            instructions="与えられた計画を確認して、より洗練させます。",  # "your task is to review the plan you receive and to refine it further."
        ),
        name="ReviseAgent",
        output_response=False,
    )
    .register_executor(
        lambda: ResponseGenerator(id="final_response_generator"),
        name="FinalResponseGenerator"
    )
    .add_edge("PlanningAgent", "ReviseAgent")
    .add_edge("ReviseAgent", "FinalResponseGenerator")
    .set_start_executor("PlanningAgent")
    .with_checkpointing(checkpoint_storage=checkpoint_storage)
)

No let's suspend workflow as follows.

In [6]:
from agent_framework import SuperStepCompletedEvent

workflow = workflow_builder.build()

async for event in workflow.run_stream("大阪の旅行計画を作成してください。"):  # "create a plan for my travel in Osaka"
    # in the first superstep completed event, we suspend the workflow
    if isinstance(event, SuperStepCompletedEvent):
        break

Let's get all checkpoints and show the number of all saved checkpoints.  
(The first checkpoint is initial checkpoint (if there are messages from initial execution) and the second is created after the first super step iteration.)

In [7]:
checkpoints = await checkpoint_storage.list_checkpoints()
print(len(checkpoints))

2


Now let's retore state by using checkpoint and resume workflow as follows.

In [8]:
workflow = workflow_builder.build()

final_checkpoint = checkpoints[-1]
events = await workflow.run(checkpoint_id=final_checkpoint.checkpoint_id)

Let's see the result (output) of this workflow. (Even after suspended, the output is correctly generated.)

In [9]:
outputs = events.get_outputs()
messages = outputs[0]
for i, (msg) in enumerate(messages, start=1):
    author_name = msg.author_name
    if author_name is None:
        author_name = "assistant" if msg.role == Role.ASSISTANT else "user"
    print("----------------------------------")
    print(f"[{i:02d}: {author_name}]")
    print(msg.text)

----------------------------------
[01: user]
大阪の旅行計画を作成してください。
----------------------------------
[02: PlanningAgent]
- **日程・目的**：旅行日数（例：1泊2日/2泊3日）とテーマ（グルメ中心・観光中心・USJ・買い物など）を決める  
- **エリア配分**：①難波/道頓堀（食・夜）②梅田（買い物・展望）③大阪城周辺（歴史）④新世界/通天閣（下町）⑤ベイエリア（海遊館/USJ）から優先順で組む  
- **モデル行程（2泊3日例）**：1日目 難波→道頓堀→心斎橋／2日目 大阪城→梅田（スカイビル等）→天満（飲み）／3日目 海遊館 or USJ→帰路  
- **移動・宿**：宿は「難波」か「梅田」だと移動が楽（地下鉄中心・ICカード推奨）。USJ重視なら「西九条/弁天町」も候補  
- **予約・予算**：USJ/海遊館・人気店は事前予約、食は「たこ焼き・串カツ・お好み焼き」を軸に。概算は交通＋宿＋入場券＋食費で組む（人数と日数で調整）
----------------------------------
[03: ReviseAgent]
- **日程・目的**：旅行日数（例：日帰り/1泊2日/2泊3日）とテーマ（グルメ・観光・USJ・買い物）を決める  
- **エリア配分**：①難波/道頓堀（食・夜）②梅田（買い物・展望）③大阪城（歴史）④新世界/通天閣（下町）⑤ベイ（海遊館/USJ）から優先順に選ぶ  
- **モデル行程（2泊3日）**：1日目 難波→道頓堀→心斎橋／2日目 大阪城→梅田（展望）→天満で飲み／3日目 海遊館 or USJ→帰路  
- **移動・宿**：宿は「難波」or「梅田」が便利（地下鉄＋ICカード推奨）。USJ重視なら「西九条/弁天町」も候補  
- **予約・予算**：USJ・海遊館・人気店は事前予約推奨。食は「たこ焼き・串カツ・お好み焼き」を軸に、交通＋宿＋入場＋食で概算を組む
